In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

INICIO = pd.Timestamp("2017-01-01")
FIM = pd.Timestamp("2018-08-31")
LIMITE = pd.Timestamp("2018-03-04")

def ler(nome, datas=()):
    df = pd.read_csv(RAW / f"olist_{nome}_dataset.csv")
    for col in datas:
        df[col] = pd.to_datetime(df[col])  # erro se algo não converter
    return df

o = ler("orders", ["order_purchase_timestamp",
                   "order_delivered_customer_date",
                   "order_estimated_delivery_date"])
c = ler("customers")
it = ler("order_items")
pg = ler("order_payments")
rv = ler("order_reviews", ["review_creation_date", "review_answer_timestamp"])

assert o["order_id"].is_unique
assert c["customer_id"].is_unique
assert o["customer_id"].isin(c["customer_id"]).all()
assert rv["review_score"].between(1, 5).all()

In [2]:
est = o["order_estimated_delivery_date"]
ent = o["order_delivered_customer_date"].dropna()
print("estimada com hora 00:00:", (est.dt.hour == 0).mean())
print("entrega com hora 00:00:", (ent.dt.hour == 0).mean())

estimada com hora 00:00: 1.0
entrega com hora 00:00: 0.029903810274057797


In [4]:
log = []

def aplicar(df, nome, mascara):
    depois = df[mascara]
    log.append({"regra": nome, "antes": len(df), "depois": len(depois),
                "removidos": len(df) - len(depois)})
    return depois

p = o.copy()
p = aplicar(p, "status = delivered", p["order_status"] == "delivered")
p = aplicar(p, "com data de entrega", p["order_delivered_customer_date"].notna())
p = aplicar(p, "entrega >= compra",
            p["order_delivered_customer_date"] >= p["order_purchase_timestamp"])
p = aplicar(p, "compra entre 2017-01-01 e 2018-08-31",
            p["order_purchase_timestamp"].dt.normalize().between(INICIO, FIM))
pd.DataFrame(log)

,regra,antes,depois,removidos
0,status = delivered,99441,96478,2963
1,com data de entrega,96478,96470,8
2,entrega >= compra,96470,96470,0
3,compra entre 2017-01-01 e 2018-08-31,96470,96203,267


In [5]:
itens = (it.groupby("order_id")
           .agg(n_itens=("order_item_id", "count"),
                valor_itens=("price", "sum"),
                frete=("freight_value", "sum"))
           .reset_index())
pagos = (pg.groupby("order_id")
           .agg(valor_pago=("payment_value", "sum"))
           .reset_index())
nota = (rv.sort_values(["order_id", "review_creation_date", "review_answer_timestamp"])
          .drop_duplicates("order_id", keep="last")[["order_id", "review_score"]]
          .rename(columns={"review_score": "nota"}))

pedidos = (p[["order_id", "customer_id", "order_purchase_timestamp",
              "order_delivered_customer_date", "order_estimated_delivery_date"]]
           .rename(columns={"order_purchase_timestamp": "compra",
                            "order_delivered_customer_date": "entrega",
                            "order_estimated_delivery_date": "estimada"})
           .merge(itens, on="order_id", how="left", validate="1:1")
           .merge(pagos, on="order_id", how="left", validate="1:1")
           .merge(nota, on="order_id", how="left", validate="1:1"))

assert pedidos["order_id"].is_unique
soma_original = it[it["order_id"].isin(pedidos["order_id"])]["price"].sum()
assert np.isclose(pedidos["valor_itens"].sum(), soma_original)
print(len(pedidos))
print(pedidos[["n_itens", "valor_pago", "nota"]].isna().sum())

96203
n_itens         0
valor_pago      0
nota          643
dtype: int64


In [6]:
pedidos["dias_atraso"] = (pedidos["entrega"].dt.normalize()
                          - pedidos["estimada"].dt.normalize()).dt.days
pedidos["atrasado"] = pedidos["dias_atraso"] > 0
print(pedidos["atrasado"].mean())
print(pedidos["dias_atraso"].describe())

0.06788769580990198
count    96203.000000
mean       -11.807969
std         10.088942
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: dias_atraso, dtype: float64


In [7]:
cli = pedidos.merge(c[["customer_id", "customer_unique_id"]],
                    on="customer_id", how="left", validate="m:1")
assert cli["customer_unique_id"].notna().all()

cli["data"] = cli["compra"].dt.normalize()
cli["primeira_data"] = cli.groupby("customer_unique_id")["data"].transform("min")
cli["dias_desde_primeiro"] = (cli["data"] - cli["primeira_data"]).dt.days

primeiro_dia = cli[cli["dias_desde_primeiro"] == 0]
clientes = (primeiro_dia.groupby("customer_unique_id")
            .agg(primeira_data=("data", "min"),
                 atrasado_primeiro=("atrasado", "max"),
                 pedidos_primeiro_dia=("order_id", "count")))

rec = (cli[cli["dias_desde_primeiro"].between(1, 180)]
       .groupby("customer_unique_id").size().rename("pedidos_180d"))
clientes = clientes.join(rec)
clientes["pedidos_180d"] = clientes["pedidos_180d"].fillna(0).astype(int)
clientes["recompra_180"] = clientes["pedidos_180d"] > 0
clientes["elegivel"] = clientes["primeira_data"].between(INICIO, LIMITE)

In [9]:
base = clientes[clientes["elegivel"]]
print("clientes totais:", len(clientes), " elegíveis:", len(base))
print("taxa de recompra (elegíveis):", base["recompra_180"].mean())
print(base.groupby("atrasado_primeiro")["recompra_180"].agg(["size", "sum"]))

pedidos.to_parquet(PROCESSED / "pedidos.parquet", index=False)
clientes.to_parquet(PROCESSED / "clientes.parquet")
cli[["order_id", "customer_unique_id", "data", "primeira_data",
     "dias_desde_primeiro", "atrasado"]].to_parquet(
    PROCESSED / "pedidos_cliente.parquet", index=False)

clientes totais: 93096  elegíveis: 56192
taxa de recompra (elegíveis): 0.019789293849658315
                    size   sum
atrasado_primeiro             
False              52234  1050
True                3958    62
